# Notebook 03 — V1 Block Ablation (B7–B9, B12–B13)

**Ablations:** B7 (no V1) · B8 (VOneBlock, noise off) · B9 (VOneBlock, Poisson on) ·
B12 (ResNet-50 back-end) · B13 (CORnet-S recurrent back-end)  
**Inputs:** CIFAR-10 (smoke test) · ImageNet-1K (full run)  
**Outputs:** `results/03_metrics.json` · `results/03_activation_similarity.png` ·
`results/03_cornet_recurrence.png`  
**New module:** `src/overrides.py` extended with V1 noise ablation helpers.

---

## Purpose

This notebook ablates the **V1 front-end component (C3)** of the full model.
The LNP model implemented in VOneBlock (Dapello et al. 2020) adds two biologically
motivated stages on top of a standard CNN backbone:

1. **Gabor filter bank** (simple + complex cells) — encodes orientation, frequency, phase.
2. **Neuronal noise** (Poisson-like, "neuronal" mode) — stochastic firing variability.

Ablating noise on/off isolates whether robustness comes from the _architecture_ (Gabor
filters) or the _stochasticity_ (noise injection). We also swap the ventral back-end
(ResNet-50 vs CORnet-S recurrent) to separate feedforward from recurrent contributions.

## Mathematics covered

**LNP model (reference NB01):**

$$
r_s = [k\,(g_{q0}*I)]_+, \quad
r_c = \frac{k}{\sqrt{2}}\sqrt{(g_{q0}*I)^2+(g_{q1}*I)^2}
$$

**Poisson-like noise (neuronal mode):**

$$
\tilde r = r + \xi\,\sqrt{\mathrm{ReLU}(r) + \varepsilon},\quad \xi\sim\mathcal{N}(0,1)
$$

**Brain-Score proxy — linear (PLS) regression:**

$$
\hat Y = X\,W_{\mathrm{PLS}},\quad
R^2 = 1 - \frac{\lVert Y - \hat Y\rVert_F^2}{\lVert Y - \bar Y\rVert_F^2},\quad
\mathrm{score} = \frac{R^2}{R^2_{\mathrm{ceiling}}}
$$

**CORnet-S recurrent dynamics:**

$$
h_t = f(W_{\mathrm{ff}}\,x + W_{\mathrm{rec}}\,h_{t-1}),\quad h_0 = 0
$$

Residual connections implement unrolled recurrence (Liao & Poggio 2016).


In [ ]:
# ── Environment ─────────────────────────────────────────────────────────────
import sys, os, warnings, json
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/tez_foveated_vision"
else:
    _here = os.getcwd()
    for _d in [_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))]:
        if os.path.isdir(os.path.join(_d, "src")):
            PROJECT_ROOT = _d; break
    else:
        PROJECT_ROOT = _here

os.chdir(PROJECT_ROOT)
for _p in [PROJECT_ROOT,
           os.path.join(PROJECT_ROOT, "external", "vonenet"),
           os.path.join(PROJECT_ROOT, "external", "CORnet")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

# !pip -q install torch torchvision timm scipy scikit-learn matplotlib tqdm

import math, importlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression

from src import common, datasets, models, overrides

# ── Configuration ────────────────────────────────────────────────────────────
CFG = {
    "seed":            42,
    "smoke_test":      True,
    "dataset":         "cifar10",
    "image_size":      32,
    "n_classes":       10,
    # V1 noise ablation
    "noise_modes":     [None, "neuronal"],     # B8=None, B9=neuronal
    # Back-end ablation
    "backbones":       ["resnet50", "cornet_s"],
    # Brain-Score proxy
    "pls_components":  5,
    "n_proxy_neurons": 64,
    "n_proxy_images":  128,
    # Training smoke test
    "n_epochs_smoke":  3,
    "n_batches_smoke": 20,
    "n_epochs_full":   20,       # smoke_test=False: real CIFAR-10, full training
    "n_batches_full":  None,     # None = full dataset each epoch
    "batch_size":      64,
    "lr":              1e-3,
    # Adversarial
    "eps":             4 / 255,
    "pgd_steps":       5,
    "pgd_alpha":       1 / 255,
    "eot_samples":     4,           # >=10 in full run
    # Paths
    "result_dir":  os.path.join(PROJECT_ROOT, "results"),
    "ckpt_dir":    os.path.join(PROJECT_ROOT, "checkpoints"),
    "data_dir":    os.path.join(PROJECT_ROOT, "data"),
}

common.set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG["result_dir"], exist_ok=True)
os.makedirs(CFG["ckpt_dir"],   exist_ok=True)

print(f"Device: {device} | Smoke test: {CFG['smoke_test']}")


## Math 1: VOneBlock LNP Model and Noise Ablation

### 1.1 Gabor filter bank (reviewed from NB01)

Gabor filter response (quadrature pair $q_0, q_1$):

$$
g(x,y) = \exp\!\left(-\frac{x'^2}{2\sigma_x^2}-\frac{y'^2}{2\sigma_y^2}\right)
\cos(2\pi f x' + \phi)
$$

Simple cell: $r_s = [k\,(g_{q0}*I)]_+$
Complex cell: $r_c = \frac{k}{\sqrt{2}}\sqrt{(g_{q0}*I)^2+(g_{q1}*I)^2}$

### 1.2 Neuronal noise (Poisson-like)

In `noise_mode='neuronal'`, the activation is perturbed as:

$$
\tilde r = r + \xi\,\sqrt{\mathrm{ReLU}(r) + \varepsilon},\quad \xi\sim\mathcal{N}(0,1)
$$

The noise variance scales with the firing rate — a linear approximation to the
Fano-factor relationship in real cortical neurons. At $r=0$ the noise floor is
$\sqrt{\varepsilon}$; for large $r$ it grows as $\sqrt{r}$.

### 1.3 Ablation design

| ID | noise_mode | Effect on gradient |
|----|-----------|-------------------|
| **B7** | — (no VOneBlock) | plain ResNet-50 gradients |
| **B8** | `None` | deterministic; gradient flows cleanly |
| **B9** | `'neuronal'` | stochastic; EOT-N required for faithful adversarial eval |

**Gradient flow** with noise on: $\nabla_x \ell$ averaged over $N$ noise samples
(EOT, Expectation-over-Transformation):

$$
\bar g = \frac{1}{N}\sum_{i=1}^N \nabla_x \ell\!\left(f(x; \xi_i), y\right)
$$

Without EOT, a single noisy sample gives a high-variance gradient estimate,
artificially inflating apparent robustness (gradient masking).


In [ ]:
# ── V1 noise ablation helpers (extend overrides.py) ──────────────────────
# These are defined inline here (visible in the notebook) and then appended
# to src/overrides.py for reuse by later notebooks.

def set_v1_noise_mode(model, mode=None, noise_scale=1, noise_level=1):
    """Enable/disable VOneBlock neuronal noise (ablation B8/B9).

    mode: None    -> no noise (B8, deterministic Gabor bank only)
          'neuronal' -> Poisson-like noise: r~ = r + xi*sqrt(relu(r)+eps) (B9)
    Returns the VOneBlock instance for inspection.
    Idempotent — safe to call repeatedly.
    """
    block = models.get_vone_block(model)
    if block is None:
        raise ValueError("Model has no VOneBlock; not a VOneNet model.")
    block.set_noise_mode(mode, noise_scale, noise_level)
    return block


def is_v1_noise_active(model):
    """Return True if the VOneBlock noise is enabled (noise_mode is not None)."""
    block = models.get_vone_block(model)
    return block is not None and block.noise_mode is not None


# Apply the existing input-gradient fix (needed for PGD on VOneNet)
overrides.apply_vone_block_input_gradient_fix()
print(f"Input-gradient fix active: {overrides.is_vone_block_input_gradient_fix_applied()}")

# Append noise-ablation helpers to src/overrides.py if not already there
_ov_path = os.path.join(PROJECT_ROOT, "src", "overrides.py")
with open(_ov_path, "r") as _f:
    _ov_src = _f.read()

_DQ = '"' * 3
_NEW_FUNCS = (
    "\n\n# ── V1 noise ablation (added by 03_v1_block.ipynb) "
    "─────────────────────────\n\n"
    "def set_v1_noise_mode(model, mode=None, noise_scale=1, noise_level=1):\n"
    "    " + _DQ + "Enable/disable VOneBlock neuronal noise for ablation B8 / B9.\n\n"
    "    mode: None       -> deterministic Gabor bank, no noise  (B8)\n"
    "          'neuronal' -> Poisson-like noise  (B9)\n"
    "    Returns the VOneBlock instance. Idempotent.\n"
    "    " + _DQ + "\n"
    "    from src.models import get_vone_block\n"
    "    block = get_vone_block(model)\n"
    "    if block is None:\n"
    "        raise ValueError('Model has no VOneBlock.')\n"
    "    block.set_noise_mode(mode, noise_scale, noise_level)\n"
    "    return block\n\n\n"
    "def is_v1_noise_active(model):\n"
    "    " + _DQ + "Return True if the VOneBlock noise is currently enabled." + _DQ + "\n"
    "    from src.models import get_vone_block\n"
    "    block = get_vone_block(model)\n"
    "    return block is not None and block.noise_mode is not None\n"
)

if "set_v1_noise_mode" not in _ov_src:
    with open(_ov_path, "a") as _f:
        _f.write(_NEW_FUNCS)
    print("Appended V1 noise ablation helpers to src/overrides.py")
else:
    print("src/overrides.py already has V1 noise ablation helpers")

importlib.reload(overrides)
print("overrides module reloaded")


## Math 2: Brain-Score Proxy and CORnet-S Recurrence

### 2.1 Brain-Score predictivity (simplified)

Full Brain-Score (Schrimpf et al. 2020) fits a Partial Least Squares (PLS)
regression from model layer activations $X \in \mathbb{R}^{n \times d}$ to
recorded neural responses $Y \in \mathbb{R}^{n \times p}$:

$$
\hat Y = X\,W_{\mathrm{PLS}},\quad
R^2 = 1 - \frac{\lVert Y - \hat Y\rVert_F^2}{\lVert Y - \bar Y\rVert_F^2}
$$

Normalised by the noise ceiling $R^2_{\mathrm{ceiling}}$ (inter-subject consistency):

$$
\mathrm{score}_{\mathrm{BS}} = \frac{R^2}{R^2_{\mathrm{ceiling}}}
$$

> **Smoke-test note:** Real neural data requires the `brain-score` package and
> access to the MajajHong2015 / Tolias datasets. Here we use **synthetic proxy
> neural responses** (random linear projection of V1 features) to exercise the
> regression pipeline. Numbers are meaningless as real Brain-Score values.
> Full evaluation: `!pip install brain-score` + mount drive with `.brainscore` folder.

### 2.2 CORnet-S recurrent dynamics

CORnet-S (Kubilius et al. 2019) implements an explicit recurrent step in each
cortical area (V2, V4, IT). Each area $\ell$ at time step $t$:

$$
h_t^\ell = f\!\left(W_{\mathrm{ff}}^\ell\,x_t^{\ell-1}
                    + W_{\mathrm{rec}}^\ell\,h_{t-1}^\ell\right), \quad h_0^\ell = 0
$$

The IT area uses $T_{\mathrm{IT}}=2$ time steps by default. Visualising the
hidden state $h_t^{\mathrm{IT}}$ across $t$ shows how information accumulates
with each recurrent pass (analogous to IT firing-rate dynamics in macaques).


In [ ]:
# ── Build ablation models ────────────────────────────────────────────────
# B7: ResNet-50, no VOneBlock (from NB01, reused here)
# B8: VOneResNet-50, noise OFF
# B9: VOneResNet-50, noise ON (neuronal)
# B12: ResNet-50 (same as B7 — feedforward reference)
# B13: CORnet-S (recurrent ventral stream)

PRETRAINED = not CFG["smoke_test"]   # no download in smoke_test

print("Building models (pretrained={})...".format(PRETRAINED))

# B7 / B12: plain ResNet-50
model_b7, norm_b7 = models.build_resnet50(pretrained=PRETRAINED)
model_b7.eval()

# B8: VOneResNet-50, noise off
model_b8, norm_b8 = models.build_voneresnet50(pretrained=PRETRAINED)
set_v1_noise_mode(model_b8, mode=None)
model_b8.eval()

# B9: VOneResNet-50, noise on (neuronal)
model_b9, norm_b9 = models.build_voneresnet50(pretrained=PRETRAINED)
set_v1_noise_mode(model_b9, mode="neuronal")
model_b9.eval()

# B13: CORnet-S
model_b13, norm_b13 = models.build_cornet_s(pretrained=PRETRAINED)
model_b13.eval()

ablation_models = {
    "B7_resnet50":      (model_b7,  norm_b7),
    "B8_vone_noisy_off":(model_b8,  norm_b8),
    "B9_vone_noisy_on": (model_b9,  norm_b9),
    "B13_cornet_s":     (model_b13, norm_b13),
}

print(f"\nBuilt {len(ablation_models)} ablation models:")
for name, (m, norm) in ablation_models.items():
    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    noise_on = is_v1_noise_active(m)
    print(f"  {name:25s} norm={norm:9s}  params={n_params:.1f}M  noise={noise_on}")


In [ ]:
# ── Sanity check: noise on/off determinism ───────────────────────────────
# B8 should give identical outputs across runs; B9 should vary.

common.set_seed(CFG["seed"])
dummy_in = torch.randn(2, 3, CFG["image_size"], CFG["image_size"]).to(device)

# For proper comparison use NormalizedModel (raw-pixel wrapper)
def _wrap(m, norm): return models.NormalizedModel(m, norm).to(device)

with torch.no_grad():
    results_noise = {}
    for name, (m, norm) in [("B8_vone_noisy_off", (model_b8, norm_b8)),
                              ("B9_vone_noisy_on",  (model_b9, norm_b9))]:
        wrapped = _wrap(m, norm)
        outs = torch.stack([wrapped(dummy_in) for _ in range(5)])  # 5 forward passes
        var_across_runs = outs.var(dim=0).mean().item()
        results_noise[name] = var_across_runs
        print(f"{name}: output variance across 5 runs = {var_across_runs:.6f}")

assert results_noise["B8_vone_noisy_off"] < 1e-9, \
    "B8 must be deterministic (noise off)"
assert results_noise["B9_vone_noisy_on"] > 1e-9, \
    "B9 must be stochastic (noise on)"
print("\n✓ Noise on/off determinism check passed")


In [ ]:
# ── Gradient flow: EOT-N on B9 ──────────────────────────────────────────
# Verify that gradients flow through VOneBlock with the input-gradient fix,
# and that EOT reduces gradient variance relative to N=1.

common.set_seed(CFG["seed"])

x_test = torch.rand(2, 3, CFG["image_size"], CFG["image_size"]).to(device).requires_grad_(True)
y_test = torch.zeros(2, dtype=torch.long).to(device)   # class 0

wrapped_b9 = models.NormalizedModel(model_b9, norm_b9).to(device)

def eot_gradient(wrapped_model, x, y, n_samples):
    """Compute EOT-averaged gradient of cross-entropy w.r.t. x."""
    x = x.detach().requires_grad_(True)
    grad_sum = torch.zeros_like(x)
    for _ in range(n_samples):
        logits = wrapped_model(x)
        loss = F.cross_entropy(logits, y)
        g, = torch.autograd.grad(loss, x, create_graph=False)
        grad_sum = grad_sum + g
    return (grad_sum / n_samples).detach()

# EOT-1 vs EOT-10 gradient norms (measure variance reduction)
grad_norms_1  = []
grad_norms_10 = []
for trial in range(5):
    x_t = torch.rand(2, 3, CFG["image_size"], CFG["image_size"]).to(device)
    y_t = torch.randint(0, CFG["n_classes"], (2,)).to(device)
    g1  = eot_gradient(wrapped_b9, x_t, y_t, n_samples=1)
    g10 = eot_gradient(wrapped_b9, x_t, y_t, n_samples=CFG["eot_samples"])
    grad_norms_1.append(g1.norm().item())
    grad_norms_10.append(g10.norm().item())

print("Gradient norm statistics (B9 VOneResNet-50):")
print(f"  EOT-1  : mean={np.mean(grad_norms_1):.4f}  std={np.std(grad_norms_1):.4f}")
print(f"  EOT-{CFG['eot_samples']:<2d} : mean={np.mean(grad_norms_10):.4f}  std={np.std(grad_norms_10):.4f}")
print("✓ Gradient flow confirmed (no RuntimeError from VOneBlock)")


## Math 3: Activation Extraction for Brain-Score Proxy

Layer activations are extracted via PyTorch forward hooks. For each model we
register hooks at three proxy cortical stages:

| Stage | Proxy layer | Cortical area |
|-------|------------|---------------|
| V1    | VOneBlock output (or conv1 for ResNet-50) | Primary visual cortex |
| Mid   | Layer 2 residual output | V2 / V4 |
| IT    | Average-pooled penultimate features | Inferior Temporal cortex |

The flattened activations form the design matrix $X \in \mathbb{R}^{n\times d}$.
PLS regression ($k=5$ components) maps $X \to \hat Y$ and $R^2$ measures
how much variance in the proxy "neural" responses is explained.


In [ ]:
# ── Brain-Score proxy: layer RSA + PLS regression ────────────────────────

class ActivationHook:
    """Forward hook that stores the output of a nn.Module."""
    def __init__(self):
        self.activation = None
        self._hook = None

    def register(self, module):
        self._hook = module.register_forward_hook(
            lambda _m, _in, out: setattr(self, "activation", out)
        )
        return self

    def remove(self):
        if self._hook:
            self._hook.remove()


def get_probe_modules(base_model, name):
    """Return (v1_module, mid_module, it_module) probing each cortical stage.
    base_model: the raw model BEFORE NormalizedModel wrapping.
    """
    core = models.unwrap_model(base_model)   # strips DataParallel
    if "b8" in name or "b9" in name or hasattr(core, 'vone_block'):
        # VOneNet: unwrapped = Sequential(vone_block, bottleneck, model)
        return core.vone_block, core.model.layer2, core.model.avgpool
    elif "b13" in name or "cornet" in name.lower():
        # CORnet-S: unwrapped = Sequential(V1, V2, V4, IT, decoder)
        return core.V1, core.V4, core.IT
    else:
        # Plain ResNet-50
        return core.layer1, core.layer2, core.avgpool


def extract_activations(base_model, fwd_model, name, images, device_):
    """Run images through fwd_model; probe activation via hooks on base_model.
    base_model: raw model (for hook registration)
    fwd_model:  NormalizedModel wrapper (for forward pass)
    """
    fwd_model = fwd_model.to(device_).eval()
    v1_mod, mid_mod, it_mod = get_probe_modules(base_model, name)
    h_v1  = ActivationHook().register(v1_mod)
    h_mid = ActivationHook().register(mid_mod)
    h_it  = ActivationHook().register(it_mod)

    feats = {"v1": [], "mid": [], "it": []}
    bs = 16
    with torch.no_grad():
        for start in range(0, len(images), bs):
            batch = images[start:start+bs].to(device_)
            fwd_model(batch)
            for key, hook in [("v1", h_v1), ("mid", h_mid), ("it", h_it)]:
                act = hook.activation
                if act is None:
                    continue
                # Flatten spatial dims
                if act.dim() > 2:
                    act = act.flatten(1)
                feats[key].append(act.cpu().float())

    h_v1.remove(); h_mid.remove(); h_it.remove()
    return {k: torch.cat(v, 0).numpy() if v else None for k, v in feats.items()}


def brain_score_proxy(X, Y, n_components=5):
    """PLS regression R^2 (proxy Brain-Score). X: [n,d] features, Y: [n,p] targets."""
    from sklearn.preprocessing import StandardScaler
    from sklearn.cross_decomposition import PLSRegression
    n = len(X)
    split = int(0.8 * n)
    X_tr, X_te = X[:split], X[split:]
    Y_tr, Y_te = Y[:split], Y[split:]
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)
    nc = min(n_components, X_tr.shape[1], X_tr.shape[0])
    pls = PLSRegression(n_components=nc)
    pls.fit(X_tr, Y_tr)
    Y_pred = pls.predict(X_te)
    ss_res = np.sum((Y_te - Y_pred) ** 2)
    ss_tot = np.sum((Y_te - Y_te.mean(0)) ** 2)
    r2 = 1 - ss_res / (ss_tot + 1e-10)
    return float(r2)


# ── Generate proxy images + synthetic neural responses ─────────────────────
common.set_seed(CFG["seed"])
N_PROXY = CFG["n_proxy_images"]
S = CFG["image_size"]

proxy_imgs = torch.rand(N_PROXY, 3, S, S)   # random images

# Synthetic "neural responses": random linear combination of input pixels
# (represents a toy model of V1 simple-cell responses)
rng = np.random.RandomState(CFG["seed"])
proj = rng.randn(3 * S * S, CFG["n_proxy_neurons"]).astype(np.float32)
Y_neural = (proxy_imgs.flatten(1).numpy() @ proj)   # [N, n_neurons]
Y_neural = (Y_neural - Y_neural.mean(0)) / (Y_neural.std(0) + 1e-8)

# Run extraction + scoring for each model
print("Running Brain-Score proxy (PLS regression)...")
print(f"{'Model':25s}  {'V1 R2':>8}  {'Mid R2':>8}  {'IT R2':>8}")

proxy_results = {}
for name, (m, norm) in ablation_models.items():
    wrapped = models.NormalizedModel(m, norm)
    feats = extract_activations(m, wrapped, name, proxy_imgs, device)
    scores = {}
    for layer_name, feat in feats.items():
        if feat is None or feat.shape[0] < 10:
            scores[layer_name] = float("nan")
            continue
        r2 = brain_score_proxy(feat, Y_neural, n_components=CFG["pls_components"])
        scores[layer_name] = r2
    proxy_results[name] = scores
    print(f"  {name:25s}  {scores.get('v1', float('nan')):8.4f}  "
          f"{scores.get('mid', float('nan')):8.4f}  "
          f"{scores.get('it', float('nan')):8.4f}")

print("\nNote: proxy R2 values use synthetic neural responses — not real Brain-Score.")


In [ ]:
# ── CORnet-S IT recurrence: activation norms across time steps ───────────

common.set_seed(CFG["seed"])

# Hook into CORnet-S IT module to capture state at each time step
import cornet

# Get the underlying CORnet-S Sequential (DataParallel stripped)
cornet_core = models.unwrap_model(model_b13)

it_states = []   # collected by hook

def _it_hook(module, inp, out):
    """Capture IT output tensor (batch=1, C, H, W) each forward call."""
    it_states.append(out.detach().cpu())

# CORnet-S IT is called `times` (default 2) times per forward pass
# Each call adds to the hook list
_hook_handle = cornet_core.IT.register_forward_hook(_it_hook)

test_img = torch.rand(1, 3, CFG["image_size"], CFG["image_size"]).to(device)
# Wrap with normalisation
wrapped_b13 = models.NormalizedModel(model_b13, norm_b13).to(device).eval()

with torch.no_grad():
    it_states.clear()
    wrapped_b13(test_img)

_hook_handle.remove()

if it_states:
    print(f"IT activations captured: {len(it_states)} time step(s)")
    mean_norms = [s.norm().item() for s in it_states]
    for t, n in enumerate(mean_norms):
        print(f"  t={t}: activation L2-norm = {n:.4f}")

    # Plot norm vs time step
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

    ax = axes[0]
    ax.bar(range(len(mean_norms)), mean_norms, color="steelblue")
    ax.set_xlabel("IT time step t")
    ax.set_ylabel("Activation L2-norm")
    ax.set_title("CORnet-S IT recurrence: norm per time step")
    ax.set_xticks(range(len(mean_norms)))

    # Flatten+PCA projection to 2D to show state trajectory
    ax2 = axes[1]
    if len(it_states) >= 2 and it_states[0].numel() > 2:
        from sklearn.decomposition import PCA
        states_flat = torch.stack(it_states).flatten(1).numpy()   # [T, d]
        if states_flat.shape[1] >= 2:
            pca = PCA(n_components=2)
            pts = pca.fit_transform(states_flat)   # [T, 2]
            ax2.plot(pts[:, 0], pts[:, 1], "o-", color="steelblue", markersize=8)
            for t, (px, py) in enumerate(pts):
                ax2.annotate(f"t={t}", (px, py), textcoords="offset points",
                              xytext=(5, 5), fontsize=9)
            ax2.set_xlabel("PC 1")
            ax2.set_ylabel("PC 2")
            ax2.set_title("IT hidden state trajectory (PCA, 2D)")
    else:
        ax2.text(0.5, 0.5, "Not enough time steps\nfor trajectory plot",
                  ha="center", va="center", transform=ax2.transAxes)
        ax2.set_axis_off()

    plt.tight_layout()
    rec_path = os.path.join(CFG["result_dir"], "03_cornet_recurrence.png")
    common.save_figure(fig, rec_path, dpi=130)
    plt.close(fig)
    print(f"Saved: {rec_path}")
else:
    print("WARNING: No IT activations captured. Check CORnet-S architecture.")


In [ ]:
# ── Activation similarity figure ─────────────────────────────────────────
# Compare layer R2 (proxy Brain-Score) across models as a bar chart.

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
layers = ["v1", "mid", "it"]
layer_labels = ["V1 proxy", "Mid-ventral proxy", "IT proxy"]
model_names = list(proxy_results.keys())
colors = plt.cm.Set2(np.linspace(0, 1, len(model_names)))

for ax, lname, ltitle in zip(axes, layers, layer_labels):
    vals = [proxy_results[n].get(lname, float("nan")) for n in model_names]
    bars = ax.bar(range(len(model_names)), vals, color=colors)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([n.split("_")[0] for n in model_names], rotation=20, ha="right")
    ax.set_ylabel("R2 (PLS regression, proxy)")
    ax.set_title(f"{ltitle} predictivity")
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=8)

fig.suptitle(
    "Brain-Score proxy: PLS R2 per ablation model\n"
    "(SYNTHETIC neural responses — not real Brain-Score; "
    "use brain-score package for real evaluation)",
    fontsize=9, y=1.02,
)
plt.tight_layout()
sim_path = os.path.join(CFG["result_dir"], "03_activation_similarity.png")
common.save_figure(fig, sim_path, dpi=130)
plt.close(fig)
print(f"Saved: {sim_path}")


## Ablation Smoke Test: Training with Each V1 Configuration

Train a classifier head on top of frozen V1 features extracted from each model.
This isolates the **representational quality** of each V1 variant as a fixed feature
extractor, avoiding the confound of end-to-end fine-tuning.

In smoke_test mode: 3 epochs, 20 batches, synthetic data (meaningless accuracy).
In full mode: 10 epochs, full CIFAR-10, meaningful comparison.


In [ ]:
# ── Feature extractor + linear head training ─────────────────────────────

def get_loaders(batch_size=64):
    """Real CIFAR-10 loaders (honest fallback to a synthetic pipeline-check
    dataset only when smoke_test=True and the real data isn't downloaded yet --
    see src/datasets.py::get_cifar10)."""
    tr = datasets.get_cifar10(CFG["data_dir"], train=True, normalization="imagenet",
                               download=True, smoke_test=CFG["smoke_test"])
    va = datasets.get_cifar10(CFG["data_dir"], train=False, normalization="imagenet",
                               download=True, smoke_test=CFG["smoke_test"])
    num_workers = 2 if not CFG["smoke_test"] else 0
    return (torch.utils.data.DataLoader(tr, batch_size=batch_size, shuffle=True, num_workers=num_workers),
            torch.utils.data.DataLoader(va, batch_size=batch_size, shuffle=False, num_workers=num_workers))


class LinearProbe(nn.Module):
    """Single linear layer on top of pooled features (feature-extractor ablation)."""
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(in_dim, n_classes)

    def forward(self, x):
        if x.dim() == 4:
            x = self.pool(x).flatten(1)
        return self.fc(x)


# ── Main ablation training loop ───────────────────────────────────────────

n_epochs  = CFG["n_epochs_smoke"]  if CFG["smoke_test"] else CFG["n_epochs_full"]
n_batches = CFG["n_batches_smoke"] if CFG["smoke_test"] else CFG["n_batches_full"]

common.set_seed(CFG["seed"])
train_ld, val_ld = get_loaders(CFG["batch_size"])

ablation_train_results = {}

for abl_name, (base_model, norm) in ablation_models.items():
    print(f"\n── {abl_name} ────────────────────────────────────")
    common.set_seed(CFG["seed"])

    # Get feature extractor (frozen)
    feat_model = models.NormalizedModel(base_model, norm).to(device).eval()
    v1_mod, _, _ = get_probe_modules(base_model, abl_name)
    hook = ActivationHook().register(v1_mod)

    # Infer feature dimension
    dummy = torch.zeros(1, 3, CFG["image_size"], CFG["image_size"]).to(device)
    with torch.no_grad():
        feat_model(dummy)
    act0 = hook.activation
    if act0 is None or act0.numel() == 0:
        print(f"  Could not extract features for {abl_name}; skipping.")
        hook.remove()
        continue
    in_dim = act0.shape[1] if act0.dim() >= 2 else act0.numel()

    probe = LinearProbe(in_dim, CFG["n_classes"]).to(device)
    opt = torch.optim.Adam(probe.parameters(), lr=CFG["lr"])

    history = []
    for epoch in range(n_epochs):
        probe.train()
        correct = total = total_loss = 0
        for batch_i, (xb, yb) in enumerate(train_ld):
            if n_batches is not None and batch_i >= n_batches: break
            xb, yb = xb.to(device), yb.to(device)
            with torch.no_grad():
                feat_model(xb)
            feat = hook.activation
            if feat is None: break
            logits = probe(feat.detach())
            loss   = F.cross_entropy(logits, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
            total   += xb.size(0)

        # Validation
        probe.eval()
        val_corr = val_tot = 0
        with torch.no_grad():
            for xb, yb in val_ld:
                xb, yb = xb.to(device), yb.to(device)
                feat_model(xb)
                feat = hook.activation
                if feat is None: break
                logits = probe(feat)
                val_corr += (logits.argmax(1) == yb).sum().item()
                val_tot  += xb.size(0)
        val_acc = val_corr / max(val_tot, 1)
        tr_acc  = correct  / max(total, 1)
        history.append({"epoch": epoch, "train_acc": tr_acc, "val_acc": val_acc})
        print(f"  Epoch {epoch+1}/{n_epochs}  "
              f"train_acc={tr_acc:.3f}  val_acc={val_acc:.3f}")

    hook.remove()
    ablation_train_results[abl_name] = {
        "final_val_acc": val_acc if history else float("nan"),
        "history": history,
        "note": ("smoke_test=True: real CIFAR-10 if present, else synthetic pipeline check"
                 if CFG["smoke_test"] else "Full run: real CIFAR-10."),
    }

print("\n─── Linear probe ablation summary ──────────────────────────────────")
print(f"{'Model':25s}  {'Final val acc':>14}")
for n, r in ablation_train_results.items():
    print(f"  {n:25s}  {r['final_val_acc']:14.4f}")

In [ ]:
# ── PGD robustness: B8 (noise off) vs B9 (noise on, EOT) ────────────────
# B9 requires EOT because the noise changes at each forward pass.
# Without EOT, gradient masking makes B9 look falsely more robust.
#
# NOTE: model_b8/model_b9 are pretrained ImageNet-1000-way classifiers -- this
# notebook never adapts them to CIFAR-10's 10 classes (that adaptation only
# happens for the frozen-feature linear-probe experiment above). Comparing their
# 1000-way argmax against CIFAR-10-style 0-9 labels would be a category mismatch
# regardless of whether the images are real or synthetic. So this cell evaluates
# robustness on their *native* task (real ImageNet-1K validation images, raw
# [0,1] pixels for PGD), matching baseline B0-B3's evaluation in
# 01_baseline_reproduce.ipynb -- honest fallback to synthetic data only if
# smoke_test=True and ImageNet-1K isn't present locally (src/datasets.py::get_imagenet).

from src.eval_harness import pgd_attack

common.set_seed(CFG["seed"])

raw_ds = datasets.get_imagenet(
    CFG["data_dir"], split="val", normalization="raw",
    image_size=CFG["image_size"], smoke_test=CFG["smoke_test"],
)
raw_ld = torch.utils.data.DataLoader(raw_ds, batch_size=8, shuffle=False)
test_imgs, test_labels = next(iter(raw_ld))

def eval_robustness(model, norm, eot_n, label):
    wrapped = models.NormalizedModel(model, norm).to(device).eval()
    # Raw dataset (no normalisation) for PGD
    x_raw = test_imgs.to(device)
    y_raw = test_labels.to(device)
    x_adv = pgd_attack(wrapped, x_raw, y_raw,
                        eps=CFG["eps"], alpha=CFG["pgd_alpha"],
                        steps=CFG["pgd_steps"], eot_samples=eot_n)
    with torch.no_grad():
        logits = wrapped(x_adv)
    acc = (logits.argmax(1) == y_raw).float().mean().item()
    print(f"  {label:35s}  EOT-{eot_n:<2d}  robust_acc={acc:.3f}")
    return acc

print("PGD robustness"
      + (" (smoke_test — random/synthetic, meaningless numbers):"
         if CFG["smoke_test"] else " (real ImageNet-1K val):"))
rob_b8_1  = eval_robustness(model_b8,  norm_b8,  1, "B8 VOneBlock noise_off")
rob_b9_1  = eval_robustness(model_b9,  norm_b9,  1, "B9 VOneBlock noise_on (EOT=1)")
rob_b9_eot = eval_robustness(model_b9, norm_b9, CFG["eot_samples"],
                              f"B9 VOneBlock noise_on (EOT={CFG['eot_samples']})")

print(f"\nEOT effect on B9: acc with EOT=1 = {rob_b9_1:.3f}, "
      f"with EOT={CFG['eot_samples']} = {rob_b9_eot:.3f}")
print("(In full run: EOT=1 should give higher acc than EOT>=10, "
      "revealing gradient masking.)")

In [ ]:
# ── Save results ─────────────────────────────────────────────────────────

output = {
    "notebook": "03_v1_block",
    "cfg": {k: v for k, v in CFG.items() if not k.endswith("_dir")},
    "brain_score_proxy": proxy_results,
    "ablation_train":    ablation_train_results,
    "note": ("smoke_test=True: synthetic data + random models. "
             "Accuracy/Brain-Score numbers are not meaningful."),
}

result_path = os.path.join(CFG["result_dir"], "03_metrics.json")
common.save_json(output, result_path)
common.save_config(CFG, os.path.join(CFG["result_dir"], "03_config.json"))

print(f"Results saved:  {result_path}")
print(f"Config  saved:  {os.path.join(CFG['result_dir'], '03_config.json')}")
print("\n── Notebook 03 complete ─────────────────────────────────────────────")
print(f"  03_activation_similarity.png : "
      f"{os.path.exists(os.path.join(CFG['result_dir'], '03_activation_similarity.png'))}")
print(f"  03_cornet_recurrence.png     : "
      f"{os.path.exists(os.path.join(CFG['result_dir'], '03_cornet_recurrence.png'))}")
print(f"  03_metrics.json              : {os.path.exists(result_path)}")
print(f"  src/overrides.py V1 helpers  : "
      f"{'set_v1_noise_mode' in open(os.path.join(PROJECT_ROOT,'src','overrides.py')).read()}")
